# Running COLMAP on a Set of Images
----
In this notebook, we'll walk through how to run COLMAP on your own set of images. Reconstructing a Gaussian Splat radiance field requires camera location and orientations and a sparse point cloud of initialization, along with the input images. COLMAP is one way to get the camera information and a sparse point cloud.
The notebook walks through a common way to run COLMAP on outdoor aerial imagery.

## Gather Data

Your reconstruction can only be as good as the data you start with, because of this it's crucial to have good images. Make sure you have good lighting and high overlap between frames. COLMAP needs to be able to match objects between frames. For outdoor scenes there are some best practices you can follow to get good results.

If you have one main object or area you're focusing on it is recommended to circle or orbit it with a camera equipped aerial vehicle. For larger outdoor scenes with no one specific focus, either overlapping circular orbits or traditional oblique mapping flight lines produce good quality results. 
<table style="width:80%;">
  <tr>
    <td style="width:33%; text-align:center;">
      <img src="images/safety_park_camera_poses.png" alt="Orbit Camera Positions" style="width:100%;">
      <div><strong>Orbit:</strong> Camera positions for a scene with a single area of focus. Cameras point towards the center of the scene wile orbiting around it. This scene is an NVIDIA collected scene known as Safety Park. </div>
    </td>
    <td style="width:33%; text-align:center;">
      <video src="images/Flight.mp4" style="width:100%;" controls></video>
      <div><strong>Oblique Camera:</strong> You can either have 5 different cameras or a smart camera to capture each of the 5 views. A ariel vehicle equipped with the camera follows the above path for a large area data capture. Arrows designate which images from the oblique to keep. Diagram adapted from <a href="https://https://enterprise-insights.dji.com/blog/smart-oblique-capture">Enterprise</a> </div>
    </td>
  </tr>
  
</table>
Be warery of motion blur if you're flying too fast the frames might have too much blur to reconstruction a high feildlety scene.

Indoor scenes
<table style="width:80%;">
  <tr>
    <td style="width:33%; text-align:center;">
      <img src="images/nerfbox.jpg" alt="Semisphere Camera Positions" style="width:100%;">
      <div><strong>Semisphere:</strong> Camera positions for a scene with a single object or small area of focus. Cameras point towards the object in placement that resembles a semisphere around the object. Diagram from <a href="https://github.com/NVlabs/instant-ngp/blob/master/docs/nerf_dataset_tips.md">nerf_dataset_tips.md</a>.</div>
    </td>
    <td style="width:33%; text-align:center;">
      <video src="images/room_video.mp4" style="width:100%;" controls></video>
      <div><strong>Indoor Scenes/rooms:</strong> If you'd like to reconstruct an interior, such as a whole room its important to move the camera location. COLMAP tends to perform poorly if the camera location doesn't move, so avoid standing still and just rotating the camera. Above is an example of camera positions in the room dataset from the <a href="https://arxiv.org/abs/2111.12077">Mip-NeRF 360</a> collection. </div>
    </td>

## From Video to Frames

Usually data is collected in the form a video, at 30 or 60 fps a short video of our scene can contain thousands of images. For a scene with just one focus, like safety park or the room, we only need between 100 - 200 images to get a good reconstruction.

We can rip frames for a video easily using [FFmpeg](https://www.ffmpeg.org/), a command line tool for processing multimedia content.

FFmpeg is available on Linux, Windows and macOS. Go to the [download](https://www.ffmpeg.org/download.html) page and follow the download and installation instructions for your OS.

We can use it to grab 1 frame per second from our input video. In this notebook we will using a video of Safety Park as our input scene.

First lets create a folder where will store our data. Lets call it safety_park. This folder will store our original input image and the output of our COLMAP run. fVDB Realty cCapture expects a the file structure will be creating, so make sure you're following similar steps if using your own data. Just change the folder and video name from safety_park to your data set name.

In [ ]:
!mkdir safety_park

Now run FFmpeg to save out frames.

In [ ]:
!ffmpeg -i safety_park.mp4 -vf "fps=1" safety_park/images_raw/frame_%04d.png

You can experiment with changing how many frames per second you save a frames. Aim to get a least 100 images from your data set. 

## Running the COLMAP Docker


[COLMAP](https://colmap.github.io/) is a SfM (structure from motion) and MVS (multiview stero) pipeline. 

